## Baseline Invoice extraction using Gemini 2.0 Flash and MLflow 3.x

In [ ]:
# ! pip install -q datasets pandas tqdm dotenv mlflow langchain-google-genai

### Imports

In [ ]:
from datasets import load_dataset
import os
import json
import time
import mlflow
from utils import (
    generate_urls,
    calculate_invoice_accuracies,
    calculate_key_level_metrics,
    calculate_individual_invoice_accuracies,
    convert_base64_to_pil,
    retrieve_token_usage,
)

from prompt import register_prompt

from dotenv import load_dotenv
from mlflow.entities import Feedback
from mlflow.genai import scorer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

load_dotenv()

### Config

In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:5000/"
MODEL_NAME = "gemini-2.5-flash"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gemini-baseline"
PROMPT_NAME = "invoice-extraction-gemini-prompt"
PROMPT_VERSION = "2"

In [ ]:
# # Set up Google API key
# import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")


### Initialize MLflow and Gemini environment

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:
# Test Gemini integration
print("Testing Gemini model initialization...")
print(f"Model: {MODEL_NAME}")
print(f"Model initialized successfully: {llm is not None}")

# Test a simple text-only query
test_message = HumanMessage(content="Hello, can you respond with 'Gemini is working'?")
try:
    test_response = llm.invoke([test_message])
    print(f"Test response: {test_response.content}")
    print("✅ Gemini integration test passed!")
except Exception as e:
    print(f"❌ Gemini integration test failed: {e}")


In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
mlflow.langchain.autolog()

### Register prompt and model

In [ ]:
register_prompt(prompt_name=PROMPT_NAME)

### Load the dataset

In [ ]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

In [ ]:
dataset["test"][0]["image"]

In [ ]:
print(dataset["test"][0]['ground_truth'])

In [ ]:
example_1 = json.loads(dataset["validation"][0]["ground_truth"])["gt_parse"]
example_2 = json.loads(dataset["validation"][1]["ground_truth"])["gt_parse"]
example_3 = json.loads(dataset["validation"][2]["ground_truth"])["gt_parse"]

### Data Preparation

In [ ]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

In [ ]:
NUM_SAMPLES = 15
test_dataset = dataset["test"].select(range(NUM_SAMPLES))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

### Inference and Evaluation

In [ ]:
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")

eval_dataset = []
for index, url in enumerate(url_list):
    eval_dict = {
        "inputs": {"image_base64": url, "schema": schema_dict},
        "expectations": {"expected_response": ground_truth_list[index]},
    }
    eval_dataset.append(eval_dict)

eval_dataset

In [ ]:
def predict_fn(image_base64, schema) -> str:
    system_prompt_template = mlflow.genai.load_prompt(
        name_or_uri=PROMPT_NAME, version=PROMPT_VERSION
    )
    if "fewshot" in PROMPT_NAME:
        system_prompt = system_prompt_template.format(
            schema=schema, example1=example_1, example2=example_2, example3=example_3
        )
    else:
        system_prompt = system_prompt_template.format(schema=schema)

    # Create message with text and image using LangChain format
    message = HumanMessage(
        content=[
            {"type": "text", "text": system_prompt},
            {
                "type": "image_url", 
                "image_url": f"data:image/jpeg;base64,{image_base64}"
            },
        ]
    )
    
    # Invoke the Gemini model
    response = llm.invoke([message])
    response_text = response.content

    return response_text

In [ ]:
# Invoke the predict_fn with the first invoice from eval_dataset
response = predict_fn(eval_dataset[0]["inputs"]["image_base64"], eval_dataset[0]["inputs"]["schema"])
response

In [ ]:
@scorer
def exact_match(inputs, outputs, expectations, trace) -> Feedback:
    try:
        outputs = json.loads(outputs.replace("```json", "").replace("```", ""))
    except Exception:
        print(f"Error parsing JSON: {outputs}")
        outputs = {}
    expectations = expectations["expected_response"]
    trace_id = trace.info.trace_id

    # Create child run for every invoice
    with mlflow.start_run(parent_run_id=parent_run.info.run_id, nested=True, run_name=trace_id):
        # Log the prediction and ground truth
        mlflow.log_dict(outputs, "prediction.json")
        mlflow.log_dict(expectations, "ground_truth.json")

        # Compute accuracy @ invoice level
        pred_df, acc = calculate_individual_invoice_accuracies(
            ground_truth=expectations, output=outputs
        )
        mlflow.log_param("trace_id", trace_id)
        mlflow.log_metric("accuracy", acc)

        # Log predictions as artifacts
        pred_df.to_csv(f"artifacts/predictions_{trace_id}.csv", index=False)
        mlflow.log_artifact(local_path=f"artifacts/predictions_{trace_id}.csv")

        # Save the invoice image
        base64_string = inputs["image_base64"]
        image = convert_base64_to_pil(base64_string)
        mlflow.log_image(image, f"input_image_{trace_id}.png")

    return Feedback(value=round(acc, 2), name="Accuracy")

In [ ]:

parent_run = mlflow.start_run(run_name=f"{MODEL_NAME}-evaluation")

start_time = time.time()

results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=[exact_match],
    predict_fn=predict_fn,
)

total_time = time.time() - start_time


In [ ]:
total_time = time.time() - start_time
# Log model details
mlflow.log_params(
    {
        "model_name": MODEL_NAME,
    }
)

mlflow.log_param("prompt", PROMPT_NAME)

# Log total number of input and output tokens - to estimate the overall cost
trace_df = mlflow.search_traces(run_id=parent_run.info.run_id)
input_tokens, output_tokens, reasoning_tokens = retrieve_token_usage(trace_df)
mlflow.log_metric("total_input_tokens", input_tokens)
mlflow.log_metric("total_output_tokens", output_tokens)
mlflow.log_metric("total_reasoning_tokens", reasoning_tokens)

# Log cost estimation
with open("cost.json", "r") as f:
    cost_dict = json.load(f)

total_input_cost = (cost_dict[MODEL_NAME]["input"] * input_tokens) / 10**6
total_output_cost = (cost_dict[MODEL_NAME]["output"] * output_tokens) / 10**6
mlflow.log_metric("estimated_cost", total_input_cost + total_output_cost)

# Log aggregated invoice metrics
response_str_list = trace_df["response"].tolist()
response_list = []
for response_str in response_str_list:
    try:
        # For Gemini responses, the content is directly available
        response_list.append(json.loads(response_str["output"]))
    except Exception:
        response_list.append("")
calculate_invoice_accuracies(response_list, ground_truth_list)
mlflow.log_artifact("artifacts/invoice_metrics.csv")


# Log aggregated key level metrics
key_level_metrics_df = calculate_key_level_metrics(response_list, ground_truth_list)
mlflow.log_artifact("artifacts/key_metrics.csv")

# Log total execution time
mlflow.log_metric("total_execution_time", total_time)

# End the parent run
mlflow.end_run()